# PersuasiX — Data Exploration

This notebook explores the PersuasiX dataset: distribution of techniques, languages, severity scores, and sample texts.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 100)

In [ ]:
# Load sample data
df = pd.read_json('../data/sample/sample_dataset.json', lines=True)
print(f'Dataset size: {len(df)} rows')
df.head()

In [ ]:
# Persuasive vs Neutral distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
persuasive_counts = df['is_persuasive'].value_counts()
axes[0].pie(persuasive_counts, labels=['Persuasive', 'Neutral'], autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[0].set_title('Persuasive vs Neutral')

# Language distribution
lang_counts = df['language'].value_counts()
lang_labels = {'en': 'English', 'fr': 'French', 'ar': 'Arabic'}
axes[1].bar([lang_labels.get(l, l) for l in lang_counts.index], lang_counts.values,
            color=['#3498db', '#e74c3c', '#2ecc71'])
axes[1].set_title('Language Distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Technique frequency
all_techniques = []
for techs in df['techniques']:
    if isinstance(techs, list):
        all_techniques.extend(techs)

tech_counts = Counter(all_techniques)

from src.utils.visualization import plot_technique_distribution
fig = plot_technique_distribution(dict(tech_counts))
plt.show()

In [ ]:
# Severity distribution
persuasive_df = df[df['is_persuasive'] == True]

if 'severity' in df.columns:
    from src.utils.visualization import plot_severity_distribution
    fig = plot_severity_distribution(df['severity'].dropna().tolist())
    plt.show()

In [ ]:
# Text length distribution
df['text_length'] = df['text'].str.len()

fig, ax = plt.subplots(figsize=(10, 5))
df.groupby('is_persuasive')['text_length'].hist(alpha=0.6, bins=20, ax=ax,
    label=['Neutral', 'Persuasive'])
ax.set_xlabel('Text Length (characters)')
ax.set_ylabel('Count')
ax.set_title('Text Length Distribution by Type')
ax.legend(['Neutral', 'Persuasive'])
plt.show()

In [ ]:
# Technique co-occurrence heatmap
from src.data.collector import TECHNIQUE_LABELS
import numpy as np

cooccurrence = np.zeros((len(TECHNIQUE_LABELS), len(TECHNIQUE_LABELS)))
for techs in df['techniques']:
    if isinstance(techs, list):
        indices = [TECHNIQUE_LABELS.index(t) for t in techs if t in TECHNIQUE_LABELS]
        for i in indices:
            for j in indices:
                cooccurrence[i][j] += 1

# Only show techniques that appear
mask = cooccurrence.sum(axis=0) > 0
labels = [l.replace('_', ' ').title() for l, m in zip(TECHNIQUE_LABELS, mask) if m]
matrix = cooccurrence[mask][:, mask]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(matrix, xticklabels=labels, yticklabels=labels, annot=True, fmt='.0f',
            cmap='YlOrRd', ax=ax)
ax.set_title('Technique Co-occurrence Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()